# Model Experimentation — ResNet50 vs HRNet × Frame Count × Bird

Compare DeepLabCut training performance across:

- **Architectures**: ResNet-50, HRNet-W32
- **Training frame counts**: 400, 800, 1400
- **Birds**: Miguel, DavidBowie, Endive

A fixed holdout of **200 frames** (per bird) is reserved with a separate seed so RMSE and training-time comparisons remain fair across all 18 runs (2 archs × 3 frame counts × 3 birds).

In [80]:
import os
import glob
import json
import logging
import random
import time
import importlib
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import sys
sys.path.insert(0, str(Path(".").resolve()))   # so DLCsupport.py is importable

import DLCsupport as dlcs
dlcs = importlib.reload(dlcs)  # pick up recent edits during notebook iteration
from DLCsupport import (
    as_posix_str,
    validate_path_exists,
    find_latest_snapshot,
    load_bodyparts,
    create_combined_project_if_missing,
    build_combined_dataset,
    set_net_type,
    create_and_train,
    ensure_config_scorer_matches_data
)

import deeplabcut
import xrommtools_copy

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

print(f"Seed        : {SEED}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"DeepLabCut  : {deeplabcut.__version__}")

Seed        : 42
NumPy       : 1.23.5
Pandas      : 2.3.3
DeepLabCut  : 3.0.0rc13


In [81]:
## 2. Global Constants
# Centralize seeds, frame budgets, architecture names, and safety toggles.

In [82]:
# Workspace root (one level above Code-Testing/)
ROOT = Path("..")

TASK         = "Canari"
EXPERIMENTER = "Tyler"

# Frame counts to evaluate — one combined project is created per (bird, n)
FRAME_COUNTS  = [200, 400, 800, 1400]

# Frames held out for evaluation. Built once per bird with HOLDOUT_SEED
# so the exact same 200 frames are used across every training comparison.
HOLDOUT_N    = 200
HOLDOUT_SEED = 999   # kept different from TRAIN_SEED on purpose
TRAIN_SEED   = SEED  # reproducible training-frame selection

# DLC PyTorch backbone names to compare
ARCHITECTURES = ["resnet_50"]

EPOCHS_SMOKE = 2
EPOCHS_FULL  = 200

# Safety toggles  — flip to True to actually run training / analysis
RUN_TRAINING      = False
RUN_VIDEO_ANALYSIS = False
USE_FULL_EPOCHS   = False

print("Global constants ready.")
print(f"  Frame counts  : {FRAME_COUNTS}")
print(f"  Architectures : {ARCHITECTURES}")
print(f"  Holdout N     : {HOLDOUT_N}  (seed {HOLDOUT_SEED})")
print(f"  Train seed    : {TRAIN_SEED}")
print(f"  Epochs        : {EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE} ({'FULL' if USE_FULL_EPOCHS else 'SMOKE'})")

Global constants ready.
  Frame counts  : [200, 400, 800, 1400]
  Architectures : ['resnet_50']
  Holdout N     : 200  (seed 999)
  Train seed    : 42
  Epochs        : 2 (SMOKE)


## 3. Per-Bird Configuration

Each bird gets its own cell with all necessary paths.
Update the date-stamped subfolder (e.g. `Canari-Tyler-2026-03-31`) if you create new single-camera projects on a different date.

In [83]:
# # -- Miguel ---------------------------------------------------------------
# MIGUEL_ROOT = ROOT / "DeepLabCut" / "Miguel"

# # Desired training sources
# MIGUEL_TRIAL_FOLDERS = [
#     MIGUEL_ROOT / "TrainingData_T6",
# ]

# # New model destination under bird root
# MIGUEL_COMBINED_BASE = MIGUEL_ROOT / "Model519"
# MIGUEL_COMBINED_BASE.mkdir(parents=True, exist_ok=True)

# # Use bird root so xma_to_dlc can discover TrainingData_T* folders
# MIGUEL_DATA_PATH = MIGUEL_ROOT
# MIGUEL_DATASET_NAME = "Miguel_T6"


# miguel_videos = sorted(MIGUEL_ROOT.rglob("*.avi"))
# MIGUEL_DUMMY_VIDEO = miguel_videos[0] if miguel_videos else MIGUEL_ROOT / "TrainingData_T6" / "missing_dummy.avi"
# MIGUEL_TEST_VIDEO_CAM1 = MIGUEL_DUMMY_VIDEO
# MIGUEL_TEST_VIDEO_CAM2 = MIGUEL_DUMMY_VIDEO

# print("Miguel paths:")
# print("  Root        :", MIGUEL_ROOT)
# print("  Trials      :", MIGUEL_TRIAL_FOLDERS)
# print("  Model base  :", MIGUEL_COMBINED_BASE)
# print("  Data path   :", MIGUEL_DATA_PATH)
# print("  Dummy video :", MIGUEL_DUMMY_VIDEO)

In [84]:
# -- DavidBowie (T15 setup) -----------------------------------------------
DAVIDBOWIE_ROOT = ROOT / "DeepLabCut2" / "DavidBowie"

# New model destination under bird root
DAVIDBOWIE_COMBINED_BASE = DAVIDBOWIE_ROOT / "Model519"
DAVIDBOWIE_COMBINED_BASE.mkdir(parents=True, exist_ok=True)


DAVIDBOWIE_T15_DATASET_NAME = "DavidBowie_T15"
# Trial-specific source folder
DAVIDBOWIE_T15_FOLDER = DAVIDBOWIE_ROOT / "TrainingData_T15"

davidbowie_15_videos = sorted(DAVIDBOWIE_T15_FOLDER.rglob("*.avi"))
DAVIDBOWIE_T15_DUMMY_VIDEO = davidbowie_15_videos[0] if davidbowie_15_videos else DAVIDBOWIE_T15_FOLDER / "missing_dummy.avi"
print("DavidBowie T15 setup:")
print("  Root        :", DAVIDBOWIE_ROOT)
print("  T15 folder  :", DAVIDBOWIE_T15_FOLDER)
print("  Model base  :", DAVIDBOWIE_COMBINED_BASE)

DavidBowie T15 setup:
  Root        : ..\DeepLabCut2\DavidBowie
  T15 folder  : ..\DeepLabCut2\DavidBowie\TrainingData_T15
  Model base  : ..\DeepLabCut2\DavidBowie\Model519


In [85]:
# -- DavidBowie (T17 setup + combined trial config) -----------------------
# Trial-specific source folder
DAVIDBOWIE_T17_FOLDER = DAVIDBOWIE_ROOT / "TrainingData_T17"

# Desired training sources (used by later trial-aware loops)
DAVIDBOWIE_TRIAL_FOLDERS = [
    DAVIDBOWIE_T15_FOLDER,
    DAVIDBOWIE_T17_FOLDER,
]

# Use bird root so xma_to_dlc can discover both TrainingData trial folders
DAVIDBOWIE_DATA_PATH = DAVIDBOWIE_ROOT
DAVIDBOWIE_T17_DATASET_NAME = "DavidBowie_T17"

davidbowie_17_videos = sorted(DAVIDBOWIE_T17_FOLDER.rglob("*.avi"))
DAVIDBOWIE_T17_DUMMY_VIDEO = davidbowie_17_videos[0] if davidbowie_17_videos else DAVIDBOWIE_T17_FOLDER / "missing_dummy.avi"
DAVIDBOWIE_TEST_VIDEO_CAM1 = DAVIDBOWIE_T17_DUMMY_VIDEO
DAVIDBOWIE_TEST_VIDEO_CAM2 = DAVIDBOWIE_T17_DUMMY_VIDEO

DAVIDBOWIE_DATASET_NAME = [DAVIDBOWIE_T15_DATASET_NAME, DAVIDBOWIE_T17_DATASET_NAME]
print("DavidBowie T17 + combined trial setup:")
print("  T17 folder  :", DAVIDBOWIE_T17_FOLDER)
print("  Trials      :", DAVIDBOWIE_TRIAL_FOLDERS)
print("  Data path   :", DAVIDBOWIE_DATA_PATH)
print("  Dummy video :", DAVIDBOWIE_T17_DUMMY_VIDEO)

DavidBowie T17 + combined trial setup:
  T17 folder  : ..\DeepLabCut2\DavidBowie\TrainingData_T17
  Trials      : [WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T15'), WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T17')]
  Data path   : ..\DeepLabCut2\DavidBowie
  Dummy video : ..\DeepLabCut2\DavidBowie\TrainingData_T17\cam1_stack.avi


In [86]:
# -- Endive ---------------------------------------------------------------
ENDIVE_ROOT = ROOT / "DeepLabCut2" / "Endive"

# Desired training source
ENDIVE_TRIAL_FOLDERS = [
    ENDIVE_ROOT / "TrainingData_T42",
]

# New model destination under bird root
ENDIVE_COMBINED_BASE = ENDIVE_ROOT / "Model519"
ENDIVE_COMBINED_BASE.mkdir(parents=True, exist_ok=True)

# Use bird root so xma_to_dlc can discover TrainingData_T* folders
ENDIVE_DATA_PATH = ENDIVE_ROOT
ENDIVE_DATASET_NAME = "Endive_T42"



endive_videos = sorted(ENDIVE_ROOT.rglob("*.avi"))
ENDIVE_DUMMY_VIDEO = endive_videos[0] if endive_videos else ENDIVE_ROOT / "TrainingData_T42" / "missing_dummy.avi"
ENDIVE_TEST_VIDEO_CAM1 = ENDIVE_DUMMY_VIDEO
ENDIVE_TEST_VIDEO_CAM2 = ENDIVE_DUMMY_VIDEO

print("Endive paths:")
print("  Root        :", ENDIVE_ROOT)
print("  Trials      :", ENDIVE_TRIAL_FOLDERS)
print("  Model base  :", ENDIVE_COMBINED_BASE)
print("  Data path   :", ENDIVE_DATA_PATH)
print("  Dummy video :", ENDIVE_DUMMY_VIDEO)

Endive paths:
  Root        : ..\DeepLabCut2\Endive
  Trials      : [WindowsPath('../DeepLabCut2/Endive/TrainingData_T42')]
  Model base  : ..\DeepLabCut2\Endive\Model519
  Data path   : ..\DeepLabCut2\Endive
  Dummy video : ..\DeepLabCut2\Endive\TrainingData_T42\cam1_stack.avi


## 4. Bird Registry
Collect all per-bird variables into one dict so the rest of the notebook can iterate without repeating code.

In [87]:
BIRDS = {
    "DavidBowie": {

        "combined_base": DAVIDBOWIE_COMBINED_BASE,
        "data_path": DAVIDBOWIE_DATA_PATH,
        "dataset_name": [DAVIDBOWIE_T15_DATASET_NAME, DAVIDBOWIE_T17_DATASET_NAME],
        "dummy_video": [DAVIDBOWIE_T15_DUMMY_VIDEO, DAVIDBOWIE_T17_DUMMY_VIDEO],
        "test_videos": [DAVIDBOWIE_T15_DUMMY_VIDEO, DAVIDBOWIE_T17_DUMMY_VIDEO],
        "trials": [15, 17],
        "trial_folders": DAVIDBOWIE_TRIAL_FOLDERS,
    },
    "Endive": {

        "combined_base": ENDIVE_COMBINED_BASE,
        "data_path": ENDIVE_DATA_PATH,
        "dataset_name": ENDIVE_DATASET_NAME,
        "dummy_video": ENDIVE_DUMMY_VIDEO,
        "test_videos": [ENDIVE_TEST_VIDEO_CAM1, ENDIVE_TEST_VIDEO_CAM2],
        "trials": [42],
        "trial_folders": ENDIVE_TRIAL_FOLDERS,
    },
    # "Miguel": {

    #     "combined_base": MIGUEL_COMBINED_BASE,
    #     "data_path": MIGUEL_DATA_PATH,
    #     "dataset_name": MIGUEL_DATASET_NAME,
    #     "dummy_video": MIGUEL_DUMMY_VIDEO,
    #     "test_videos": [MIGUEL_TEST_VIDEO_CAM1, MIGUEL_TEST_VIDEO_CAM2],
    #     "trials": [6],
    #     "trial_folders": MIGUEL_TRIAL_FOLDERS,
    # },
}

print(f"Registered {len(BIRDS)} birds: {list(BIRDS)}")
for bird_name, bird_cfg in BIRDS.items():
    print(f"  {bird_name}: trials={bird_cfg['trials']}")

Registered 2 birds: ['DavidBowie', 'Endive']
  DavidBowie: trials=[15, 17]
  Endive: trials=[42]


In [88]:
BIRDS['DavidBowie']

{'combined_base': WindowsPath('../DeepLabCut2/DavidBowie/Model519'),
 'data_path': WindowsPath('../DeepLabCut2/DavidBowie'),
 'dataset_name': ['DavidBowie_T15', 'DavidBowie_T17'],
 'dummy_video': [WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T15/cam1_stack.avi'),
  WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T17/cam1_stack.avi')],
 'test_videos': [WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T15/cam1_stack.avi'),
  WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T17/cam1_stack.avi')],
 'trials': [15, 17],
 'trial_folders': [WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T15'),
  WindowsPath('../DeepLabCut2/DavidBowie/TrainingData_T17')]}

## 5. Create Combined Projects

One combined DLC project is created for **each (bird, frame-count) pair** — 9 training projects total.
A separate **holdout project** (200 frames) is also created per bird so evaluation frames are never in any training set.

All 12 projects are stored in `combined_configs`, keyed by `(bird_name, nframes)` or `(bird_name, "holdout")`.

In [89]:
# Reuse BIRDS from Cell 10 (trial-aware registry).
# This cell keeps compatibility with the original notebook flow.
print("Trial-aware BIRDS registry ready:")
for bird_name, bird_cfg in BIRDS.items():
    print(f"  {bird_name}: trials={bird_cfg['trials']} | base={bird_cfg['combined_base']}")

Trial-aware BIRDS registry ready:
  DavidBowie: trials=[15, 17] | base=..\DeepLabCut2\DavidBowie\Model519
  Endive: trials=[42] | base=..\DeepLabCut2\Endive\Model519


In [90]:
# combined_configs[(bird_name, nframes, trial)] -> Path to config.yaml (training projects)
combined_configs: dict = {}

for bird_name, bird_cfg in BIRDS.items():
    base = bird_cfg["combined_base"]
    

    for trial in bird_cfg["trials"]:
        dummy = bird_cfg["dummy_video"]
        if isinstance(dummy, list):
            logging.warning(f"{bird_name} has a list of dummy videos. Using the appropriate one for trial {trial}.")
            dummy = next(x for x in dummy if str(trial) in str(x))
            print(f"  Using dummy video {dummy} for trial {trial} config creation.")
        for n in FRAME_COUNTS:
            if bird_name == "Miguel" and n == 1400:
                logging.info(f"Skipping Miguel trial {trial} for n={n} due to lack of data.")
                continue
            project_name = f"n{n}_T{trial}"
            cfg_path = create_combined_project_if_missing(
                task=TASK,
                experimenter=EXPERIMENTER,
                combined_project_root=base / project_name,
                dummy_video=dummy,
            )
            combined_configs[(bird_name, n, trial)] = cfg_path

    print(
        f"{bird_name}: {len(FRAME_COUNTS) * len(bird_cfg['trials'])} training projects ready "
        f"({len(FRAME_COUNTS)} frame-counts x {len(bird_cfg['trials'])} trials)."
    )

print(f"\nTotal combined configs registered: {len(combined_configs)}")
for (bird_name, nframes, trial), path in sorted(combined_configs.items()):
    print(f"  {bird_name:12s} n={nframes:>4} T{trial:<3} -> {path}")

print("\nNext: run the bodyparts setup cell to write bird-specific bodyparts into each config.")

INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n400_T15\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n800_T15\Canari-Tyler-2026-05-22\config.yaml


  Using dummy video ..\DeepLabCut2\DavidBowie\TrainingData_T15\cam1_stack.avi for trial 15 config creation.
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\videos"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\labeled-data"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\training-datasets"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\dlc-models"
Attempting to create a symbolic link of the video ...
Symlink creation impossible (exFat architecture?): copying the video instead.
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\TrainingData_T15\c

INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n1400_T15\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T17\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n400_T17\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n800_T17\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n1400_T17\Canari-Tyler-2026-05-22\config.yaml


Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n1400_T15\Canari-Tyler-2026-05-22\config.yaml"

A new project with name Canari-Tyler-2026-05-22 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n1400_T15 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to your project's needs.
 Once you have changed the configuration file, use the function 'extract_frames' to select frames for labeling.
. [OPTIONAL] Use the function 'add_new_videos' to add new videos to your project (at any stage).
  Using dummy video ..\DeepLabCut2\DavidBowie\TrainingData_T17\cam1_stack.avi for trial 17 config creation.
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T17\Canari-Tyler-2026-05-22\videos"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\

INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n400_T42\Canari-Tyler-2026-05-22\config.yaml


DavidBowie: 8 training projects ready (4 frame-counts x 2 trials).
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\videos"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\labeled-data"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\training-datasets"
Created "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\dlc-models"
Attempting to create a symbolic link of the video ...
Symlink creation impossible (exFat architecture?): copying the video instead.
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\TrainingData_T42\cam1_stack.avi copied to C:\Users\Salle-Cineradio\Documents\Ma

INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42\Canari-Tyler-2026-05-22\config.yaml
INFO:DLCsupport:Created combined project: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n1400_T42\Canari-Tyler-2026-05-22\config.yaml


C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\TrainingData_T42\cam1_stack.avi copied to C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42\Canari-Tyler-2026-05-22\videos\cam1_stack.avi
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42\Canari-Tyler-2026-05-22\videos\cam1_stack.avi
Generated "C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42\Canari-Tyler-2026-05-22\config.yaml"

A new project with name Canari-Tyler-2026-05-22 is created at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42 and a configurable file (config.yaml) is stored there. Change the parameters in this file to adapt to your project's needs.
 Once you have changed the configuration file, use the function 'extract_frames' to select fra

## 5B. Apply Bird-Specific Bodyparts to Configs

This step writes each bird's bodyparts list from `DLCsupport.BIRD_BODYPARTS` into all training project configs for that bird.

Update bodypart lists in `DLCsupport.py` as needed, then run this cell.

In [91]:
APPLY_BIRD_BODYPARTS = True

bodyparts_configs_by_bird = {
    bird_name: [
        combined_configs[(bird_name, n, trial)]
        for trial in bird_cfg["trials"]
        for n in FRAME_COUNTS if not (bird_name == "Miguel" and n == 1400)  # Miguel doesn't have enough frames for n=1400
    ]
    for bird_name, bird_cfg in BIRDS.items()
}

if not APPLY_BIRD_BODYPARTS:
    print("Skipping bodyparts write step. Set APPLY_BIRD_BODYPARTS = True to enable.")
else:
    bodyparts_apply_df = dlcs.apply_bird_bodyparts_to_configs(
        bodyparts_configs_by_bird,
        strict=True,
    )
    display(bodyparts_apply_df)

    failed = bodyparts_apply_df[bodyparts_apply_df["status"] != "ok"]
    if not failed.empty:
        raise RuntimeError(
            "Bodyparts write verification failed for one or more configs. "
            "Check DLCsupport.BIRD_BODYPARTS and rerun."
        )

    print("Bird-specific bodyparts have been written to all training configs.")

,bird,config,status,n_bodyparts
0,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
1,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
2,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
3,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
4,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
5,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
6,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
7,DavidBowie,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,21
8,Endive,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,24
9,Endive,C:\Users\Salle-Cineradio\Documents\MachineLear...,ok,24


Bird-specific bodyparts have been written to all training configs.


## 6. Build Combined Datasets

Build **9 training datasets** (3 birds × 3 frame counts) plus **3 holdout datasets** (one per bird, 200 frames each).

**Holdout strategy**:  
- Holdout frames are sampled with `HOLDOUT_SEED = 999`.  
- Training frames are sampled with `TRAIN_SEED = 42`.  
- Because the seeds differ, overlap between holdout and training pools is negligible. The same holdout frames will be produced identically every time this cell is re-run.

In [92]:
summary_rows = [
    {
        "bird": bird,
        "nframes": n,
        "trial": trial,
        "config": str(cfg),
    }
    for (bird, n, trial), cfg in sorted(combined_configs.items())
]

combined_configs_df = pd.DataFrame(summary_rows)
display(combined_configs_df)
print(f"Total trial-aware configs: {len(combined_configs_df)}")

,bird,nframes,trial,config
0,DavidBowie,200,15,C:\Users\Salle-Cineradio\Documents\MachineLear...
1,DavidBowie,200,17,C:\Users\Salle-Cineradio\Documents\MachineLear...
2,DavidBowie,400,15,C:\Users\Salle-Cineradio\Documents\MachineLear...
3,DavidBowie,400,17,C:\Users\Salle-Cineradio\Documents\MachineLear...
4,DavidBowie,800,15,C:\Users\Salle-Cineradio\Documents\MachineLear...
5,DavidBowie,800,17,C:\Users\Salle-Cineradio\Documents\MachineLear...
6,DavidBowie,1400,15,C:\Users\Salle-Cineradio\Documents\MachineLear...
7,DavidBowie,1400,17,C:\Users\Salle-Cineradio\Documents\MachineLear...
8,Endive,200,42,C:\Users\Salle-Cineradio\Documents\MachineLear...
9,Endive,400,42,C:\Users\Salle-Cineradio\Documents\MachineLear...


Total trial-aware configs: 12


In [93]:
dataset_build_log = []   # records what was (or would be) built

for bird_name, bird_cfg in BIRDS.items():
    trial_folders = bird_cfg.get("trial_folders", [])
    if len(trial_folders) != len(bird_cfg["trials"]):
        raise ValueError(
            f"{bird_name}: trial_folders length ({len(trial_folders)}) must match trials length ({len(bird_cfg['trials'])})."
        )

    trial_to_folder = {
        trial_id: trial_folder
        for trial_id, trial_folder in zip(bird_cfg["trials"], trial_folders)
    }

    for trial in bird_cfg["trials"]:
        trial_data_path = trial_to_folder[trial]
        for n in FRAME_COUNTS:
            if bird_name == "Miguel" and n == 1400:
                logging.info(f"Skipping Miguel trial {trial} for n={n} due to lack of data.")
                continue
            train_cfg = combined_configs[(bird_name, n, trial)]
            print(
                f"[{bird_name}] Building training dataset (T{trial}, n={n} frames, seed={TRAIN_SEED}) "
                f"from {trial_data_path}"
            )
            if isinstance(bird_cfg['dataset_name'], list):
                dataset_name = next(x for x in bird_cfg['dataset_name'] if str(trial) in str(x))
                logging.info(f"{bird_name} has multiple dataset names. Using '{dataset_name}' for trial {trial}.")
            else:
                dataset_name = bird_cfg['dataset_name']
            try:
                build_combined_dataset(
                    combined_config=train_cfg,
                    data_path=trial_data_path,
                    dataset_name=f"n{n}",
                    experimenter=EXPERIMENTER,
                    nframes=n,
                    frame_selection_seed=TRAIN_SEED,
                )
                dataset_build_log.append(
                    {
                        "bird": bird_name,
                        "trial": trial,
                        "split": "train",
                        "nframes": n,
                        "data_path": str(trial_data_path),
                        "config": str(train_cfg),
                    }
                )
            except Exception as e:
                logging.error(
                    f"Failed to build dataset for {bird_name} T{trial} n={n}: {e}"
                )

print(f"\n{'─'*60}")
print(f"Dataset builds complete: {len(dataset_build_log)} training")
if dataset_build_log:
    print(
        pd.DataFrame(dataset_build_log)[
            ["bird", "trial", "split", "nframes", "data_path"]
        ].to_string(index=False)
    )

INFO:root:DavidBowie has multiple dataset names. Using 'DavidBowie_T15' for trial 15.
INFO:DLCsupport:Using xrommtools_copy.xma_to_dlc for dataset build
INFO:DLCsupport:Frame selection seed set to 42


[DavidBowie] Building training dataset (T15, n=200 frames, seed=42) from ..\DeepLabCut2\DavidBowie\TrainingData_T15
Using trial folders: ['TrainingData_T15']
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO:root:DavidBowie has multiple dataset names. Using 'DavidBowie_T15' for trial 15.
INFO:DLCsupport:Using xrommtools_copy.xma_to_dlc for dataset build
INFO:DLCsupport:Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[DavidBowie] Building training dataset (T15, n=400 frames, seed=42) from ..\DeepLabCut2\DavidBowie\TrainingData_T15
Using trial folders: ['TrainingData_T15']
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO:root:DavidBowie has multiple dataset names. Using 'DavidBowie_T15' for trial 15.
INFO:DLCsupport:Using xrommtools_copy.xma_to_dlc for dataset build
INFO:DLCsupport:Frame selection seed set to 42


...

...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[Endive] Building training dataset (T42, n=400 frames, seed=42) from ..\DeepLabCut2\Endive\TrainingData_T42
Using trial folders: ['TrainingData_T42']
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO:DLCsupport:Using xrommtools_copy.xma_to_dlc for dataset build
INFO:DLCsupport:Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[Endive] Building training dataset (T42, n=800 frames, seed=42) from ..\DeepLabCut2\Endive\TrainingData_T42
Using trial folders: ['TrainingData_T42']
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...


INFO:DLCsupport:Using xrommtools_copy.xma_to_dlc for dataset build
INFO:DLCsupport:Frame selection seed set to 42


...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset
[Endive] Building training dataset (T42, n=1400 frames, seed=42) from ..\DeepLabCut2\Endive\TrainingData_T42
Using trial folders: ['TrainingData_T42']
Extracting camera 1 trial images and 2D points...
Extracting camera 2 trial images and 2D points...
...done.
Training data extracted to projectpath/labeled-data. Now use deeplabcut.create_training_dataset

────────────────────────────────────────────────────────────
Dataset builds complete: 12 training
      bird  trial split  nframes                                  data_path
DavidBowie     15 train      200 ..\DeepLabCut2\DavidBowie\TrainingData_T15
DavidBowie     15 train      400 ..\DeepLabCut2\DavidBowie\TrainingData_T15
DavidBowie     15 train      800 ..\DeepLabCut2\DavidBowie\TrainingData_T15
DavidBowie     15 train     1400 ..\DeepLabCut2\DavidBowie\TrainingData_T15
DavidBowie     17 train      200 ..\DeepLabCut2\DavidBowie\T

In [94]:
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
dataset_frame_counts = []

for (bird_name, nframes, trial), config_path in sorted(
    combined_configs.items(),
    key=lambda item: (item[0][0], int(item[0][1]), int(item[0][2])),
):
    labeled_dir = Path(config_path).parent / "labeled-data"
    image_paths = [
        p for p in labeled_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    unique_image_files = {
        p.relative_to(labeled_dir).as_posix()
        for p in image_paths
    }

    # Collapse cam1/cam2 pairs to one logical frame ID when possible.
    unique_logical_frames = {
        p.stem.replace("_cam1_", "_cam_").replace("_cam2_", "_cam_")
        for p in image_paths
    }

    dataset_frame_counts.append({
        "bird": bird_name,
        "trial": trial,
        "nframes": nframes,
        "unique_image_files": len(unique_image_files),
        "unique_logical_frames": len(unique_logical_frames),
        "labeled_data_dir": str(labeled_dir),
    })

dataset_frame_counts_df = pd.DataFrame(dataset_frame_counts).sort_values(
    ["bird", "trial", "nframes"]
)

print(
    dataset_frame_counts_df[
        ["bird", "trial", "nframes", "unique_image_files", "unique_logical_frames"]
    ].to_string(index=False)
)

dataset_frame_counts_df

      bird  trial  nframes  unique_image_files  unique_logical_frames
DavidBowie     15      200                 400                    200
DavidBowie     15      400                 800                    400
DavidBowie     15      800                1600                    800
DavidBowie     15     1400                2800                   1400
DavidBowie     17      200                 400                    200
DavidBowie     17      400                 800                    400
DavidBowie     17      800                1600                    800
DavidBowie     17     1400                2800                   1400
    Endive     42      200                 400                    200
    Endive     42      400                 800                    400
    Endive     42      800                1600                    800
    Endive     42     1400                2800                   1400


,bird,trial,nframes,unique_image_files,unique_logical_frames,labeled_data_dir
0,DavidBowie,15,200,400,200,C:\Users\Salle-Cineradio\Documents\MachineLear...
2,DavidBowie,15,400,800,400,C:\Users\Salle-Cineradio\Documents\MachineLear...
4,DavidBowie,15,800,1600,800,C:\Users\Salle-Cineradio\Documents\MachineLear...
6,DavidBowie,15,1400,2800,1400,C:\Users\Salle-Cineradio\Documents\MachineLear...
1,DavidBowie,17,200,400,200,C:\Users\Salle-Cineradio\Documents\MachineLear...
3,DavidBowie,17,400,800,400,C:\Users\Salle-Cineradio\Documents\MachineLear...
5,DavidBowie,17,800,1600,800,C:\Users\Salle-Cineradio\Documents\MachineLear...
7,DavidBowie,17,1400,2800,1400,C:\Users\Salle-Cineradio\Documents\MachineLear...
8,Endive,42,200,400,200,C:\Users\Salle-Cineradio\Documents\MachineLear...
9,Endive,42,400,800,400,C:\Users\Salle-Cineradio\Documents\MachineLear...


In [95]:
#Verification
for bird_name, bird_cfg in BIRDS.items():
        for trial in bird_cfg["trials"]:
            for n in FRAME_COUNTS:
                if bird_name == "Miguel" and n == 1400:
                    logging.info(f"Skipping Miguel trial {trial} for n={n} due to lack of data.")
                    continue
                config_path = combined_configs[(bird_name, n, trial)]
                print(
                    f"[{bird_name}] Verifying dataset for T{trial} n={n} frames at {config_path} ..."
                )
                try:
                    ensure_config_scorer_matches_data(config_path)
                    print(f"  ✓ Config scorer matches labeled data for {bird_name} T{trial} n={n}.")
                except Exception as e:
                    logging.error(
                        f"  ✗ Config-data mismatch for {bird_name} T{trial} n={n}: {e}"
                    )
                    

[DavidBowie] Verifying dataset for T15 n=200 frames at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\config.yaml ...
  ✓ Config scorer matches labeled data for DavidBowie T15 n=200.
[DavidBowie] Verifying dataset for T15 n=400 frames at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n400_T15\Canari-Tyler-2026-05-22\config.yaml ...
  ✓ Config scorer matches labeled data for DavidBowie T15 n=400.
[DavidBowie] Verifying dataset for T15 n=800 frames at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n800_T15\Canari-Tyler-2026-05-22\config.yaml ...
  ✓ Config scorer matches labeled data for DavidBowie T15 n=800.
[DavidBowie] Verifying dataset for T15 n=1400 frames at C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n1400_T15\Can

# Examining les DonnèeS

In [ ]:
# Cell 28: CSV/H5 <-> image alignment audit/fix + interactive labeled-frame viewer
import re
from pathlib import Path
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# Safety switches
AUDIT_AND_FIX_ALL = False   # False = audit only, True = rewrite CollectedData indices where resolvable
FAIL_IF_MISSING = False     # True = raise if unresolved rows remain after optional fixes

def _find_collected_data_files(labeled_dir: Path) -> tuple[list[Path], list[Path]]:
    h5_files = sorted(labeled_dir.glob("**/CollectedData_*.h5"))
    csv_files = sorted(labeled_dir.glob("**/CollectedData_*.csv"))
    return h5_files, csv_files

def _read_collected_data(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".h5":
        return pd.read_hdf(path)
    return pd.read_csv(path, header=[0, 1, 2], index_col=0)

def _write_collected_data(df: pd.DataFrame, path: Path) -> None:
    if path.suffix.lower() == ".h5":
        # Keep DLC-compatible key name used by most projects.
        df.to_hdf(path, key="df_with_missing", mode="w")
    else:
        df.to_csv(path)

def _normalize_stem(stem: str) -> str:
    out = stem
    out = out.replace("Trep_UND", "")
    out = re.sub(r"_+", "_", out).strip("_")
    out = re.sub(r"^(TrainingData)_T\d+_(cam[12]_\d+)$", r"\1_\2", out)
    out = re.sub(r"(_T\d+)_T\d+_", r"\1_", out)
    return out

def _normalize_index_string(idx_str: str) -> str:
    s = idx_str.replace("\\", "/")
    s = re.sub(r"/+/", "/", s).strip()
    return s

def _build_image_lookups(image_paths: list[Path]) -> tuple[dict[str, list[Path]], dict[str, list[Path]]]:
    exact = {}
    normalized = {}
    for p in image_paths:
        exact.setdefault(p.name, []).append(p)
        norm_name = f"{_normalize_stem(p.stem)}{p.suffix.lower()}"
        normalized.setdefault(norm_name, []).append(p)
    return exact, normalized

def _resolve_index_to_image(
    project_dir: Path,
    labeled_dir: Path,
    idx_value: object,
    exact_name_lookup: dict[str, list[Path]],
    normalized_name_lookup: dict[str, list[Path]],
    image_paths: list[Path],
 ) -> Path | None:
    raw = _normalize_index_string(str(idx_value))
    p = Path(raw)

    candidates = [project_dir / p, labeled_dir / p, labeled_dir / p.name]
    if p.parts and p.parts[0].lower() == "labeled-data" and len(p.parts) > 1:
        candidates.append(labeled_dir / Path(*p.parts[1:]))

    for c in candidates:
        if c.exists() and c.is_file() and c.suffix.lower() in IMAGE_EXTS:
            return c

    # Exact basename fallback
    if p.name in exact_name_lookup and exact_name_lookup[p.name]:
        return exact_name_lookup[p.name][0]

    # Normalized basename fallback
    norm_name = f"{_normalize_stem(Path(p.name).stem)}{Path(p.name).suffix.lower()}"
    if norm_name in normalized_name_lookup and normalized_name_lookup[norm_name]:
        return normalized_name_lookup[norm_name][0]

    # Last fallback: linear name match (case-insensitive)
    lname = p.name.lower()
    for ip in image_paths:
        if ip.name.lower() == lname:
            return ip

    return None

def _canonical_index_from_image(labeled_dir: Path, image_path: Path) -> str:
    rel = image_path.relative_to(labeled_dir).as_posix()
    # DLC index convention starts at labeled-data/...
    return f"labeled-data/{rel}"

def audit_and_optionally_fix_collected_data(
    config_path: Path,
    apply_fixes: bool = False,
 ) -> dict:
    project_dir = Path(config_path).parent
    labeled_dir = project_dir / "labeled-data"
    if not labeled_dir.exists():
        return {
            "project": str(project_dir),
            "rows": 0,
            "images": 0,
            "missing": 0,
            "rewritable": 0,
            "rewritten": 0,
            "status": "missing-labeled-data",
        }

    h5_files, csv_files = _find_collected_data_files(labeled_dir)
    source_path = h5_files[0] if h5_files else (csv_files[0] if csv_files else None)
    if source_path is None:
        return {
            "project": str(project_dir),
            "rows": 0,
            "images": 0,
            "missing": 0,
            "rewritable": 0,
            "rewritten": 0,
            "status": "missing-collecteddata",
        }

    df = _read_collected_data(source_path)
    image_paths = [p for p in labeled_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    exact_name_lookup, normalized_name_lookup = _build_image_lookups(image_paths)

    new_index = []
    missing = 0
    rewritable = 0

    for idx in df.index:
        resolved = _resolve_index_to_image(
            project_dir=project_dir,
            labeled_dir=labeled_dir,
            idx_value=idx,
            exact_name_lookup=exact_name_lookup,
            normalized_name_lookup=normalized_name_lookup,
            image_paths=image_paths,
        )
        if resolved is None:
            missing += 1
            new_index.append(str(idx))
            continue

        canonical = _canonical_index_from_image(labeled_dir, resolved)
        old_norm = _normalize_index_string(str(idx))
        if old_norm != canonical:
            rewritable += 1
        new_index.append(canonical)

    rewritten = 0
    if apply_fixes and rewritable > 0:
        fixed_df = df.copy()
        fixed_df.index = pd.Index(new_index, name=df.index.name)

        # Write source format
        _write_collected_data(fixed_df, source_path)
        rewritten = rewritable

        # Keep sibling format in sync if present
        sibling_h5 = source_path.with_suffix(".h5")
        sibling_csv = source_path.with_suffix(".csv")
        if source_path.suffix.lower() == ".h5" and sibling_csv.exists():
            _write_collected_data(fixed_df, sibling_csv)
        if source_path.suffix.lower() == ".csv" and sibling_h5.exists():
            _write_collected_data(fixed_df, sibling_h5)

    status = "ok" if missing == 0 else "missing-images"
    if apply_fixes and rewritten > 0:
        status = "fixed" if missing == 0 else "fixed-with-missing"

    return {
        "project": str(project_dir),
        "rows": int(len(df)),
        "images": int(len(image_paths)),
        "missing": int(missing),
        "rewritable": int(rewritable),
        "rewritten": int(rewritten),
        "status": status,
    }

def audit_all_projects(combined_cfgs: dict, apply_fixes: bool = False) -> pd.DataFrame:
    rows = []
    for (_, _, _), cfg in sorted(combined_cfgs.items(), key=lambda item: (item[0][0], int(item[0][2]), int(item[0][1]))):
        rows.append(audit_and_optionally_fix_collected_data(Path(cfg), apply_fixes=apply_fixes))
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["status", "missing", "rewritable", "project"], ascending=[True, False, False, True])
    return out

def _build_color_map(bodyparts: list[str]) -> dict[str, tuple]:
    cmap = plt.colormaps.get_cmap("tab20")
    return {bp: cmap(i % cmap.N) for i, bp in enumerate(bodyparts)}

def _load_best_collected_data(labeled_dir: Path) -> tuple[Path | None, pd.DataFrame | None, str | None]:
    h5_files, csv_files = _find_collected_data_files(labeled_dir)
    candidates = h5_files + csv_files
    if not candidates:
        return None, None, "No CollectedData_*.h5 or .csv found."
    for c in candidates:
        try:
            df = _read_collected_data(c)
            if isinstance(df.columns, pd.MultiIndex):
                return c, df, None
        except Exception:
            continue
    return None, None, "Could not load a valid DLC MultiIndex CollectedData file."

def _build_bp_coord_column_map(df: pd.DataFrame, bodyparts: list[str]) -> dict[str, dict[str, list]]:
    mapping: dict[str, dict[str, list]] = {}
    for bp in bodyparts:
        x_cols = []
        y_cols = []
        for col in df.columns:
            parts = col if isinstance(col, tuple) else (col,)
            part_strs = [str(p) for p in parts]
            if bp not in part_strs:
                continue

            coord = None
            for p in reversed(part_strs):
                lp = str(p).lower()
                if lp in {"x", "y"}:
                    coord = lp
                    break

            if coord == "x":
                x_cols.append(col)
            elif coord == "y":
                y_cols.append(col)

        mapping[bp] = {"x": x_cols, "y": y_cols}
    return mapping

def _first_finite_from_cols(row: pd.Series, cols: list) -> float | None:
    for c in cols:
        try:
            v = pd.to_numeric(row[c], errors="coerce")
        except Exception:
            continue
        if pd.notna(v) and np.isfinite(float(v)):
            return float(v)
    return None

def _apply_zoom(ax, img_shape, zoom, cx, cy):
    h, w = img_shape[0], img_shape[1]
    if w <= 0 or h <= 0:
        return

    zoom = max(1.0, float(zoom))
    cx = float(np.clip(cx, 0, max(w - 1, 0)))
    cy = float(np.clip(cy, 0, max(h - 1, 0)))

    half_w = w / (2.0 * zoom)
    half_h = h / (2.0 * zoom)

    x0 = max(0.0, cx - half_w)
    x1 = min(float(w - 1), cx + half_w)
    y0 = max(0.0, cy - half_h)
    y1 = min(float(h - 1), cy + half_h)

    ax.set_xlim(x0, x1)
    # Keep image top-left origin convention.
    ax.set_ylim(y1, y0)

def create_labeled_frame_plotter(bird_name: str, trial: int, nframes: int, config_path: Path):
    project_dir = Path(config_path).parent
    labeled_dir = project_dir / "labeled-data"
    if not labeled_dir.exists():
        return widgets.Label(f"Missing labeled-data directory: {labeled_dir}")

    image_paths = [p for p in labeled_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    exact_name_lookup, normalized_name_lookup = _build_image_lookups(image_paths)

    data_file, df, err = _load_best_collected_data(labeled_dir)
    if err:
        return widgets.Label(f"{err} under: {labeled_dir}")
    if df is None or df.empty:
        return widgets.Label(f"No rows found in {data_file.name if data_file else 'CollectedData file'}")

    try:
        bodyparts = list(df.columns.get_level_values("bodyparts").unique())
    except Exception:
        try:
            bodyparts = list(df.columns.get_level_values(1).unique())
        except Exception as e:
            return widgets.Label(f"Could not infer bodyparts from columns: {e}")

    bp_coord_map = _build_bp_coord_column_map(df, bodyparts)
    color_map = _build_color_map(bodyparts)

    frame_slider = widgets.IntSlider(
        value=0, min=0, max=len(df) - 1, step=1, description="Frame",
        continuous_update=False, layout=Layout(width="380px"),
    )
    marker_size_slider = widgets.FloatSlider(
        value=45.0, min=10.0, max=180.0, step=5.0, description="Point size",
        continuous_update=False, readout_format=".1f", layout=Layout(width="320px"),
    )
    zoom_slider = widgets.FloatSlider(
        value=1.0, min=1.0, max=12.0, step=0.25, description="Zoom",
        continuous_update=False, readout_format=".2f", layout=Layout(width="300px"),
    )
    center_x_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=1.0, description="Center X",
        continuous_update=False, readout_format=".0f", layout=Layout(width="320px"),
    )
    center_y_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=1.0, description="Center Y",
        continuous_update=False, readout_format=".0f", layout=Layout(width="320px"),
    )
    show_labels_toggle = widgets.Checkbox(value=False, description="Show labels")
    show_legend_toggle = widgets.Checkbox(value=False, description="Show legend")
    out = widgets.Output(layout=Layout(border="1px solid #BBB", padding="8px"))

    def _sync_pan_sliders(img):
        h, w = img.shape[0], img.shape[1]
        target_x_max = float(max(w - 1, 0))
        target_y_max = float(max(h - 1, 0))

        if center_x_slider.max != target_x_max:
            center_x_slider.min = 0.0
            center_x_slider.max = target_x_max
            center_x_slider.step = 1.0
            center_x_slider.value = target_x_max / 2.0

        if center_y_slider.max != target_y_max:
            center_y_slider.min = 0.0
            center_y_slider.max = target_y_max
            center_y_slider.step = 1.0
            center_y_slider.value = target_y_max / 2.0

    def render(*_):
        with out:
            clear_output(wait=True)
            row_idx = frame_slider.value
            row = df.iloc[row_idx]
            idx_value = df.index[row_idx]

            img_path = _resolve_index_to_image(
                project_dir=project_dir,
                labeled_dir=labeled_dir,
                idx_value=idx_value,
                exact_name_lookup=exact_name_lookup,
                normalized_name_lookup=normalized_name_lookup,
                image_paths=image_paths,
            )

            if img_path is None:
                print(f"Image not found for index: {idx_value}")
                return

            img = plt.imread(str(img_path))
            _sync_pan_sliders(img)

            fig, ax = plt.subplots(figsize=(10, 7))
            ax.imshow(img)

            plotted = 0
            for bp in bodyparts:
                x = _first_finite_from_cols(row, bp_coord_map[bp]["x"])

                y = _first_finite_from_cols(row, bp_coord_map[bp]["y"])
                if x is None or y is None:
                    continue
                if not (np.isfinite(x) and np.isfinite(y)):
                    continue

                ax.scatter(
                    x, y, s=marker_size_slider.value, color=color_map[bp],
                    edgecolors="white", linewidths=0.6, alpha=0.95, label=bp,
                )
                plotted += 1
                if show_labels_toggle.value:
                    ax.text(
                        x + 2, y + 2, bp, fontsize=8, color=color_map[bp],
                        bbox=dict(facecolor="white", alpha=0.45, edgecolor="none", pad=0.2),
                    )

            _apply_zoom(
                ax=ax,
                img_shape=img.shape,
                zoom=zoom_slider.value,
                cx=center_x_slider.value,
                cy=center_y_slider.value,
            )

            if show_legend_toggle.value and plotted > 0:
                handles, labels = ax.get_legend_handles_labels()
                unique = {}
                for h, l in zip(handles, labels):
                    if l not in unique:
                        unique[l] = h
                ax.legend(
                    unique.values(), unique.keys(),
                    loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0, fontsize=8,
                )

            ax.set_title(
                f"{bird_name} | T{trial} | n={nframes} | frame {row_idx + 1}/{len(df)} | points={plotted} | zoom={zoom_slider.value:.2f}x\n{img_path.name}",
                fontsize=11,
            )
            ax.set_axis_off()
            plt.tight_layout()
            plt.show()

            if plotted == 0:
                print("No finite x/y points found for this frame with current CollectedData columns.")

    frame_slider.observe(render, names="value")
    marker_size_slider.observe(render, names="value")
    zoom_slider.observe(render, names="value")
    center_x_slider.observe(render, names="value")
    center_y_slider.observe(render, names="value")
    show_labels_toggle.observe(render, names="value")
    show_legend_toggle.observe(render, names="value")

    controls_row_1 = widgets.HBox([frame_slider, marker_size_slider, zoom_slider])
    controls_row_2 = widgets.HBox([center_x_slider, center_y_slider, show_labels_toggle, show_legend_toggle])
    controls = widgets.VBox([controls_row_1, controls_row_2])

    render()
    return widgets.VBox([controls, out])

# 1) Audit / optional fix of CSV-H5 index alignment against actual image files
alignment_df = audit_all_projects(combined_configs, apply_fixes=AUDIT_AND_FIX_ALL)
if alignment_df.empty:
    print("No combined configs found for alignment audit.")
else:
    print("Alignment summary:")
    display(alignment_df[["status", "missing", "rewritable", "rewritten", "rows", "images", "project"]])

    total_missing = int(alignment_df["missing"].sum())
    print(f"Total unresolved index rows: {total_missing}")
    if total_missing > 0 and FAIL_IF_MISSING:
        raise RuntimeError("Unresolved CSV/H5 index rows remain. Inspect alignment_df before training.")

# 2) Viewer tabs (uses robust resolver even when some rows are stale)
tab_children = []
tab_titles = []
for (bird_name, nframes, trial), config_path in sorted(
    combined_configs.items(),
    key=lambda item: (item[0][0], int(item[0][2]), int(item[0][1])),
):
    viewer = create_labeled_frame_plotter(bird_name, trial, nframes, Path(config_path))
    tab_children.append(viewer)
    tab_titles.append(f"{bird_name} T{trial} n={nframes}")

tab = widgets.Tab(children=tab_children)
for i, title in enumerate(tab_titles):
    tab.set_title(i, title)

display(tab)
print("Viewer ready. Use zoom and center sliders to inspect labels closely.")
print("Set AUDIT_AND_FIX_ALL=True in this same cell to rewrite resolvable index paths.")

Alignment summary:


,status,missing,rewritable,rewritten,rows,images,project
3,ok,0,0,0,2800,2800,C:\Users\Salle-Cineradio\Documents\MachineLear...
7,ok,0,0,0,2800,2800,C:\Users\Salle-Cineradio\Documents\MachineLear...
0,ok,0,0,0,400,400,C:\Users\Salle-Cineradio\Documents\MachineLear...
4,ok,0,0,0,400,400,C:\Users\Salle-Cineradio\Documents\MachineLear...
1,ok,0,0,0,800,800,C:\Users\Salle-Cineradio\Documents\MachineLear...
5,ok,0,0,0,800,800,C:\Users\Salle-Cineradio\Documents\MachineLear...
2,ok,0,0,0,1600,1600,C:\Users\Salle-Cineradio\Documents\MachineLear...
6,ok,0,0,0,1600,1600,C:\Users\Salle-Cineradio\Documents\MachineLear...
11,ok,0,0,0,2800,2800,C:\Users\Salle-Cineradio\Documents\MachineLear...
8,ok,0,0,0,400,400,C:\Users\Salle-Cineradio\Documents\MachineLear...


Total unresolved index rows: 0


Viewer ready. Use zoom and center sliders to inspect labels closely.
Set AUDIT_AND_FIX_ALL=True in this same cell to rewrite resolvable index paths.


## 6.5 Creating DLC Training Network

In [100]:
training_results = []  # list of dicts — one entry per (bird, trial, n, arch) run

review_configs_by_bird = {
    bird_name: [
        combined_configs[(bird_name, n, trial)]
        for trial in bird_cfg["trials"]
        for n in FRAME_COUNTS
        if (bird_name, n, trial) in combined_configs
    ]
    for bird_name, bird_cfg in BIRDS.items()
}

for bird_name, bird_cfg in BIRDS.items():
    for trial in bird_cfg["trials"]:
        for n in FRAME_COUNTS:
            if (bird_name, n, trial) not in combined_configs:
                logging.warning(
                    f"Skipping dataset Creation record for {bird_name} T{trial} n={n} due to missing config."
                )
                continue
            train_cfg = combined_configs[(bird_name, n, trial)]
            for arch in ARCHITECTURES:
                run_label = f"{bird_name} | T{trial} | n={n} | {arch}"
                print(f"\n{'─'*60}")
                print(f"Training: {run_label}")

                set_net_type(train_cfg, arch)

                t0 = time.perf_counter()
                try:
                    print(train_cfg)
                    deeplabcut.create_training_dataset(train_cfg)
                    elapsed = time.perf_counter() - t0
                    
                except Exception as exc:
                    
                    elapsed = time.perf_counter() - t0
                    print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Training runs recorded: {len(training_results)}")

INFO:root:Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Training: DavidBowie | T15 | n=200 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\config.yaml
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\labeled-data\cam1_stack\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\DavidBowie\\Model519\\n200_T15\\Canari-Tyler-2026-05-22\\labeled-data\\cam1_stack', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\DavidBowie\\Model519\\n200_T15\\Canari-Tyler-2026-05-22\\labeled-data\\n200']
C:\Users\Salle

INFO:root:Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead
INFO:root:Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Training: DavidBowie | T15 | n=800 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n800_T15\Canari-Tyler-2026-05-22\config.yaml
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n800_T15\Canari-Tyler-2026-05-22\labeled-data\cam1_stack\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\DavidBowie\\Model519\\n800_T15\\Canari-Tyler-2026-05-22\\labeled-data\\cam1_stack', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\DavidBowie\\Model519\\n800_T15\\Canari-Tyler-2026-05-22\\labeled-data\\n800']
C:\Users\Salle

INFO:root:Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


...


────────────────────────────────────────────────────────────
Training: Endive | T42 | n=200 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\config.yaml
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\labeled-data\cam1_stack\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\Endive\\Model519\\n200_T42\\Canari-Tyler-2026-05-22\\labeled-data\\cam1_stack', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\Endive\\Model519\\n200_T42\\Canari-Tyler-2026-05-22\\labeled-data\\n200']
C:\Users\Salle-Cineradio\Documents

INFO:root:Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Training: Endive | T42 | n=800 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42\Canari-Tyler-2026-05-22\config.yaml
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42\Canari-Tyler-2026-05-22\labeled-data\cam1_stack\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\Endive\\Model519\\n800_T42\\Canari-Tyler-2026-05-22\\labeled-data\\cam1_stack', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\Endive\\Model519\\n800_T42\\Canari-Tyler-2026-05-22\\labeled-data\\n800']
C:\Users\Salle-Cineradio\Documents

INFO:root:Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Training: Endive | T42 | n=1400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n1400_T42\Canari-Tyler-2026-05-22\config.yaml
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n1400_T42\Canari-Tyler-2026-05-22\labeled-data\cam1_stack\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\Endive\\Model519\\n1400_T42\\Canari-Tyler-2026-05-22\\labeled-data\\cam1_stack', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut2\\Endive\\Model519\\n1400_T42\\Canari-Tyler-2026-05-22\\labeled-data\\n1400']
C:\Users\Salle-Cineradio\Doc

## 7. Training — ResNet50

Iterates over all **12 combinations** (1 architectures × 4 frame counts × 2 birds (3Trial)).  

Wall-clock training time is recorded per run for comparison.

In [67]:
RUN_TRAINING = True
USE_FULL_EPOCHS = True
epochs = EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE

for bird_name, bird_cfg in BIRDS.items():
    for trial in bird_cfg["trials"]:
        for n in FRAME_COUNTS[0:3]:
            train_cfg = combined_configs[(bird_name, n, trial)]
            for arch in ARCHITECTURES:
                run_label = f"{bird_name} | T{trial} | n={n} | {arch}"
                print(f"\n{'─'*60}")
                print(f"Training: {run_label}")

                set_net_type(train_cfg, arch)

                t0 = time.perf_counter()
                try:
                    print(train_cfg)
                    latest_snap = deeplabcut.train_network(train_cfg, epochs=epochs)
                    elapsed = time.perf_counter() - t0
                    training_results.append(
                        {
                            "bird": bird_name,
                            "trial": trial,
                            "nframes": n,
                            "architecture": arch,
                            "epochs": epochs,
                            "trained": True,
                            "train_time_s": round(elapsed, 1),
                            "latest_snapshot": latest_snap,
                            "notes": "",
                        }
                    )
                    print(f"  Done in {elapsed/60:.1f} min  ->  {latest_snap}")
                except Exception as exc:
                    elapsed = time.perf_counter() - t0
                    training_results.append(
                        {
                            "bird": bird_name,
                            "trial": trial,
                            "nframes": n,
                            "architecture": arch,
                            "epochs": epochs,
                            "trained": False,
                            "train_time_s": round(elapsed, 1),
                            "latest_snapshot": None,
                            "notes": str(exc),
                        }
                    )
                    print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Training runs recorded: {len(training_results)}")
pd.DataFrame(training_results)[["bird", "trial", "nframes", "architecture", "trained", "train_time_s"]]

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
inference:
  multithreading:
    enabled: True
    queue_length: 4
    timeout: 30.0
  compile:
    enabled: False
    backend: inductor
  autocast:
    enabled: False
metadata:
  project_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22
  pose_config_path: C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\dlc-models-pytorch\iteration-0\CanariMay22-trainset95shuffle1\train\pytorch_config.yaml
  bodyparts: ['Cranium_Le


────────────────────────────────────────────────────────────
Training: DavidBowie | T15 | n=200 | resnet_50
..\DeepLabCut2\DavidBowie\Model519\n200_T15\Canari-Tyler-2026-05-22\config.yaml


Loading pretrained weights from Hugging Face hub (timm/resnet50_gn.a1h_in1k)
HTTP Request: HEAD https://huggingface.co/timm/resnet50_gn.a1h_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[timm/resnet50_gn.a1h_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
Data Transforms:
  Training:   Compose([
  Affine(always_apply=False, p=0.5, interpolation=1, mask_interpolation=0, cval=0, mode=0, scale={'x': (0.5, 1.25), 'y': (0.5, 1.25)}, translate_percent=None, translate_px={'x': (0, 0), 'y': (0, 0)}, rotate=(-30, 30), fit_output=False, shear={'x': (0.0, 0.0), 'y': (0.0, 0.0)}, cval_mask=0, keep_ratio=True, rotate_method='largest_box'),
  PadIfNeeded(always_apply=True, p=1.0, min_height=448, min_width=448, pad_height_divisor=None, pad_width_divisor=None, position=PositionType.CENTER, border_mode=0, value=None, mask_value=None),
  KeypointAwareCrop(always_apply=True, p=1.0, width=448, height=448, max_shift=0.1, crop_

  Done in 0.6 min  ->  None

────────────────────────────────────────────────────────────
Training: DavidBowie | T15 | n=400 | resnet_50
..\DeepLabCut2\DavidBowie\Model519\n400_T15\Canari-Tyler-2026-05-22\config.yaml


Loading pretrained weights from Hugging Face hub (timm/resnet50_gn.a1h_in1k)
HTTP Request: HEAD https://huggingface.co/timm/resnet50_gn.a1h_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[timm/resnet50_gn.a1h_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
Data Transforms:
  Training:   Compose([
  Affine(always_apply=False, p=0.5, interpolation=1, mask_interpolation=0, cval=0, mode=0, scale={'x': (0.5, 1.25), 'y': (0.5, 1.25)}, translate_percent=None, translate_px={'x': (0, 0), 'y': (0, 0)}, rotate=(-30, 30), fit_output=False, shear={'x': (0.0, 0.0), 'y': (0.0, 0.0)}, cval_mask=0, keep_ratio=True, rotate_method='largest_box'),
  PadIfNeeded(always_apply=True, p=1.0, min_height=448, min_width=448, pad_height_divisor=None, pad_width_divisor=None, position=PositionType.CENTER, border_mode=0, value=None, mask_value=None),
  KeypointAwareCrop(always_apply=True, p=1.0, width=448, height=448, max_shift=0.1, crop_

...

Loading pretrained weights from Hugging Face hub (timm/resnet50_gn.a1h_in1k)
HTTP Request: HEAD https://huggingface.co/timm/resnet50_gn.a1h_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[timm/resnet50_gn.a1h_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
Data Transforms:
  Training:   Compose([
  Affine(always_apply=False, p=0.5, interpolation=1, mask_interpolation=0, cval=0, mode=0, scale={'x': (0.5, 1.25), 'y': (0.5, 1.25)}, translate_percent=None, translate_px={'x': (0, 0), 'y': (0, 0)}, rotate=(-30, 30), fit_output=False, shear={'x': (0.0, 0.0), 'y': (0.0, 0.0)}, cval_mask=0, keep_ratio=True, rotate_method='largest_box'),
  PadIfNeeded(always_apply=True, p=1.0, min_height=448, min_width=448, pad_height_divisor=None, pad_width_divisor=None, position=PositionType.CENTER, border_mode=0, value=None, mask_value=None),
  KeypointAwareCrop(always_apply=True, p=1.0, width=448, height=448, max_shift=0.1, crop_

  Done in 1.3 min  ->  None

────────────────────────────────────────────────────────────
Training: DavidBowie | T17 | n=800 | resnet_50
..\DeepLabCut2\DavidBowie\Model519\n800_T17\Canari-Tyler-2026-05-22\config.yaml


Loading pretrained weights from Hugging Face hub (timm/resnet50_gn.a1h_in1k)
HTTP Request: HEAD https://huggingface.co/timm/resnet50_gn.a1h_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[timm/resnet50_gn.a1h_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
Data Transforms:
  Training:   Compose([
  Affine(always_apply=False, p=0.5, interpolation=1, mask_interpolation=0, cval=0, mode=0, scale={'x': (0.5, 1.25), 'y': (0.5, 1.25)}, translate_percent=None, translate_px={'x': (0, 0), 'y': (0, 0)}, rotate=(-30, 30), fit_output=False, shear={'x': (0.0, 0.0), 'y': (0.0, 0.0)}, cval_mask=0, keep_ratio=True, rotate_method='largest_box'),
  PadIfNeeded(always_apply=True, p=1.0, min_height=448, min_width=448, pad_height_divisor=None, pad_width_divisor=None, position=PositionType.CENTER, border_mode=0, value=None, mask_value=None),
  KeypointAwareCrop(always_apply=True, p=1.0, width=448, height=448, max_shift=0.1, crop_

  Done in 2.5 min  ->  None

────────────────────────────────────────────────────────────
Training: Endive | T42 | n=200 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n200_T42\Canari-Tyler-2026-05-22\config.yaml
  ERROR: Could not find a shuffle with trainingset fraction 0.95 and index 1

────────────────────────────────────────────────────────────
Training: Endive | T42 | n=400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n400_T42\Canari-Tyler-2026-05-22\config.yaml
  ERROR: Could not find a shuffle with trainingset fraction 0.95 and index 1

────────────────────────────────────────────────────────────
Training: Endive | T42 | n=800 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut2\Endive\Model519\n800_T42\Canari-Tyler-2026-05-22\config.yaml
  ERROR: Could not find a shuffle with trainingset fraction 0.

,bird,trial,nframes,architecture,trained,train_time_s
0,DavidBowie,15,200,resnet_50,True,0.2
1,DavidBowie,15,400,resnet_50,True,0.2
2,DavidBowie,15,800,resnet_50,True,0.4
3,DavidBowie,15,1400,resnet_50,True,0.6
4,DavidBowie,17,200,resnet_50,True,0.2
5,DavidBowie,17,400,resnet_50,True,0.2
6,DavidBowie,17,800,resnet_50,True,0.4
7,DavidBowie,17,1400,resnet_50,True,0.6
8,Endive,42,200,resnet_50,False,0.1
9,Endive,42,400,resnet_50,False,0.1


In [ ]:
RUN_TRAINING = True
USE_FULL_EPOCHS = True
epochs = EPOCHS_FULL if USE_FULL_EPOCHS else EPOCHS_SMOKE
training_results = []  # list of dicts — one entry per (bird, trial, n, arch) run

if not RUN_TRAINING:
    print("DRY RUN — set RUN_TRAINING = True to execute training.")
    for bird_name, bird_cfg in BIRDS.items():
        for trial in bird_cfg["trials"]:
            for n in FRAME_COUNTS:
                for arch in ARCHITECTURES:
                    training_results.append(
                        {
                            "bird": bird_name,
                            "trial": trial,
                            "nframes": n,
                            "architecture": arch,
                            "epochs": epochs,
                            "trained": False,
                            "train_time_s": None,
                            "latest_snapshot": None,
                            "notes": "dry run",
                        }
                    )
else:
    review_configs_by_bird = {
        bird_name: [
            combined_configs[(bird_name, n, trial)]
            for trial in bird_cfg["trials"]
            for n in FRAME_COUNTS
        ]
        for bird_name, bird_cfg in BIRDS.items()
    }
    dlcs.require_bodyparts_review_before_training(
        run_training=RUN_TRAINING,
        configs_by_bird=review_configs_by_bird,
    )

    for bird_name, bird_cfg in BIRDS.items():
        for trial in bird_cfg["trials"]:
            for n in FRAME_COUNTS:
                train_cfg = combined_configs[(bird_name, n, trial)]
                for arch in ARCHITECTURES:
                    run_label = f"{bird_name} | T{trial} | n={n} | {arch}"
                    print(f"\n{'─'*60}")
                    print(f"Training: {run_label}")

                    set_net_type(train_cfg, arch)

                    t0 = time.perf_counter()
                    try:
                        latest_snap = dlcs.create_and_train(
                            config_path=train_cfg,
                            epochs=epochs,
                            snapshot_path=None,
                        )
                        elapsed = time.perf_counter() - t0
                        training_results.append(
                            {
                                "bird": bird_name,
                                "trial": trial,
                                "nframes": n,
                                "architecture": arch,
                                "epochs": epochs,
                                "trained": True,
                                "train_time_s": round(elapsed, 1),
                                "latest_snapshot": latest_snap,
                                "notes": "",
                            }
                        )
                        print(f"  Done in {elapsed/60:.1f} min  ->  {latest_snap}")
                    except Exception as exc:
                        elapsed = time.perf_counter() - t0
                        training_results.append(
                            {
                                "bird": bird_name,
                                "trial": trial,
                                "nframes": n,
                                "architecture": arch,
                                "epochs": epochs,
                                "trained": False,
                                "train_time_s": round(elapsed, 1),
                                "latest_snapshot": None,
                                "notes": str(exc),
                            }
                        )
                        print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Training runs recorded: {len(training_results)}")
pd.DataFrame(training_results)[["bird", "trial", "nframes", "architecture", "trained", "train_time_s"]]


Review bodyparts per bird before training:

Bird: DavidBowie
  configured bodyparts (21): ['Cranium_Left_Anterior', 'Cranium_Left_Middle', 'Cranium_Right_Middle', 'Cranium_Right_Posterior', 'Beak_Mandible_Left_Posterior', 'Beak_Mandibile_Left_Anterior', 'Beak_Mandible_Right_Anterior', 'Beak_Maxilla_Right_Anterior', 'Beak_Maxilla_Right_Posterior', 'Beak_Maxilla_Left_Anterior', 'Beak_Maxilla_Left_Posterior', 'Tongue_Lower_Palette', 'Glottis', 'Superior_Trachea', 'Keel_Dorsal_Anterior', 'Keel_Dorsal_Posterior', 'Keel_Inferior', 'Pelvis_Right_Inferior', 'Pelvis_Right_Superior', 'Pelvis_Left_Superior', 'Pelvis_Left_Inferior']
  C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\config.yaml
    bodyparts (21): ['Cranium_Left_Anterior', 'Cranium_Left_Middle', 'Cranium_Right_Middle', 'Cranium_Right_Posterior', 'Beak_Mandible_Left_Posterior', 'Beak_Mandibile_Left_Anterior', 'Beak_Mandib

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


All bird confirmations accepted. Training can proceed.

────────────────────────────────────────────────────────────
Training: DavidBowie | n=200 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n200\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\labeled-data\\Cam1', 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\labeled-data\\Canari']
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MN

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n200\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n400\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n400\\Canari-Tyler-2026-04-21\\labeled

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n400\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=800 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n800\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n800\\Canari-Tyler-2026-04-21\\labeled

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n800\\Canari-Tyler-2026-04-21\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr21\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Training: DavidBowie | n=1400 | resnet_50
C:\Users\Salle-Cineradio\Documents\MachineLearning\BirdSongs-MNHN\Testing\DeepLabCut\DavidBowie\UpdatedModelBuilds\Canari_combined_n1400\Canari-Tyler-2026-04-21\labeled-data\Cam1\CollectedData_Tyler.h5  not found (perhaps not annotated).
Annotation data was not found by splitting video paths (from config['video_sets']). An alternative route is taken...
The following folders were found: ['C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\UpdatedModelBuilds\\Canari_combined_n1400\\Canari-Tyler-2026-04-21\\labe

,bird,nframes,architecture,trained,train_time_s
0,DavidBowie,200,resnet_50,False,0.2
1,DavidBowie,400,resnet_50,False,0.3
2,DavidBowie,800,resnet_50,False,0.6
3,DavidBowie,1400,resnet_50,False,0.9


## 7B. Create Combined Projects for Augmentation Runs

Create dedicated combined projects for augmentation experiments so model outputs are organized under augmentation-specific project folders.

In [15]:
# Dedicated config registries for augmentation experiments.
augmentation_combined_configs: dict = {}  # Legacy (bird, n) registry
augmentation_variant_configs: dict = {}   # New (bird, n, augmentation) registry

AUGMENTATION_FRAME_COUNTS = [200, 400, 800]
AUGMENTATION_VARIANTS = [
    {
        "name": "flip_lr",
        "modelprefix": "flip_lr",
    },
    {
        "name": "flip_lr_sharp_blur",
        "modelprefix": "flip_lr_sharp_blur",
    },
]

for bird_name, bird_cfg in BIRDS.items():
    base = bird_cfg["combined_base"]
    dummy = bird_cfg["dummy_video"]

    for n in AUGMENTATION_FRAME_COUNTS:
        for variant in AUGMENTATION_VARIANTS:
            variant_name = variant["name"]
            cfg_path = create_combined_project_if_missing(
                task=TASK,
                experimenter=EXPERIMENTER,
                combined_project_root=base / f"Canari_augexp_{variant_name}_n{n}",
                dummy_video=dummy,
            )
            augmentation_variant_configs[(bird_name, n, variant_name)] = cfg_path

        # Keep compatibility with any older references that use (bird, n).
        augmentation_combined_configs[(bird_name, n)] = augmentation_variant_configs[
            (bird_name, n, AUGMENTATION_VARIANTS[0]["name"])
        ]

print(
    f"Created/reused {len(augmentation_variant_configs)} augmentation project configs "
    f"({len(AUGMENTATION_VARIANTS)} variants x {len(AUGMENTATION_FRAME_COUNTS)} frame counts x {len(BIRDS)} birds)."
)
for (bird_name, n, variant_name), path in sorted(augmentation_variant_configs.items()):
    print(f"  {bird_name:12s}  n={n:>4}  aug={variant_name:18s}  ->  {path}")



# Ensure bird-specific bodyparts are written to all augmentation-specific configs.
augmentation_bodyparts_by_bird = {
    bird_name: [
        augmentation_variant_configs[(bird_name, n, variant["name"])]
        for n in AUGMENTATION_FRAME_COUNTS
        for variant in AUGMENTATION_VARIANTS
    ]
    for bird_name in BIRDS
}
augmentation_bodyparts_df = dlcs.apply_bird_bodyparts_to_configs(
    augmentation_bodyparts_by_bird,
    strict=True,
)
display(augmentation_bodyparts_df)


INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n200\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n200\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n400\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n400\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n800\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n800\Canari-Tyler-2026-04-17\config.yaml
INFO | Reusing existing combined project: ..\DeepLabCut\Endive\TestData\Canari_augexp_flip_lr_n200\Canari-Tyler-2026-04-17\config.yam

Created/reused 12 augmentation project configs (2 variants x 3 frame counts x 2 birds).
  DavidBowie    n= 200  aug=flip_lr             ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n200\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 200  aug=flip_lr_sharp_blur  ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n200\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 400  aug=flip_lr             ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n400\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 400  aug=flip_lr_sharp_blur  ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n400\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 800  aug=flip_lr             ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_n800\Canari-Tyler-2026-04-17\config.yaml
  DavidBowie    n= 800  aug=flip_lr_sharp_blur  ->  ..\DeepLabCut\DavidBowie\TestData\Canari_augexp_flip_lr_sharp_blur_n800\Canari-Tyler-2026-04-

,bird,config,status,n_bodyparts
0,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
1,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
2,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
3,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
4,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
5,DavidBowie,..\DeepLabCut\DavidBowie\TestData\Canari_augex...,ok,21
6,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24
7,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24
8,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24
9,Endive,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...,ok,24


## 7B. Adding training datasets

In [16]:

# Populate each augmentation-specific project with combined labeled data (same Step 6 pattern).
augmentation_dataset_build_log = []
for bird_name, bird_cfg in BIRDS.items():
    for n in AUGMENTATION_FRAME_COUNTS:
        for variant in AUGMENTATION_VARIANTS:
            variant_name = variant["name"]
            train_cfg = augmentation_variant_configs[(bird_name, n, variant_name)]
            print(
                f"[{bird_name}] Building augmentation dataset "
                f"(aug={variant_name}, n={n} frames, seed={TRAIN_SEED}) ..."
            )
            build_combined_dataset(
                combined_config=train_cfg,
                data_path=bird_cfg["data_path"],
                dataset_name=bird_cfg["dataset_name"],
                experimenter=EXPERIMENTER,
                nframes=n,
                frame_selection_seed=TRAIN_SEED,
            )
            augmentation_dataset_build_log.append({
                "bird": bird_name,
                "augmentation": variant_name,
                "nframes": n,
                "config": str(train_cfg),
            })

print(f"\n{'─'*60}")
print(f"Augmentation dataset builds complete: {len(augmentation_dataset_build_log)}")
if augmentation_dataset_build_log:
    display(pd.DataFrame(augmentation_dataset_build_log))



INFO | Frame selection seed set to 42


[DavidBowie] Building augmentation dataset (aug=flip_lr, n=200 frames, seed=42) ...


UnboundLocalError: local variable 'camera' referenced before assignment

In [17]:
augmentation_variant_configs

{('DavidBowie',
  200,
  'flip_lr'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_n200/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  200,
  'flip_lr_sharp_blur'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_sharp_blur_n200/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  400,
  'flip_lr'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_n400/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  400,
  'flip_lr_sharp_blur'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_sharp_blur_n400/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  800,
  'flip_lr'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_n800/Canari-Tyler-2026-04-17/config.yaml'),
 ('DavidBowie',
  800,
  'flip_lr_sharp_blur'): WindowsPath('../DeepLabCut/DavidBowie/TestData/Canari_augexp_flip_lr_sharp_blur_n800/Canari-Tyler-2026-04-17/config.yaml'),
 ('Endive',
  200,
  'flip_lr'): W

In [18]:
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
dataset_frame_counts = []

for (bird_name, split_key, variant_name), config_path in sorted(
    augmentation_variant_configs.items(),
    key=lambda item: (item[0][0], str(item[0][1])),
):
    labeled_dir = Path(config_path).parent / "labeled-data"
    image_paths = [
        p for p in labeled_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_exts
    ]

    unique_image_files = {
        p.relative_to(labeled_dir).as_posix()
        for p in image_paths
    }

    # Collapse cam1/cam2 pairs to one logical frame ID when possible.
    unique_logical_frames = {
        p.stem.replace("_cam1_", "_cam_").replace("_cam2_", "_cam_")
        for p in image_paths
    }

    dataset_frame_counts.append({
        "bird": bird_name,
        "split": "holdout" if split_key == "holdout" else "train",
        "nframes": HOLDOUT_N if split_key == "holdout" else split_key,
        "variant": variant_name,
        "unique_image_files": len(unique_image_files),
        "unique_logical_frames": len(unique_logical_frames),
        "labeled_data_dir": str(labeled_dir),
    })

dataset_frame_counts_df = pd.DataFrame(dataset_frame_counts).sort_values(
    ["bird", "split", "nframes", "variant"]
)

print(
    dataset_frame_counts_df[
        ["bird", "split", "nframes", "variant", "unique_image_files", "unique_logical_frames"]
    ].to_string(index=False)
)

dataset_frame_counts_df

      bird split  nframes            variant  unique_image_files  unique_logical_frames
DavidBowie train      200            flip_lr                 400                    200
DavidBowie train      200 flip_lr_sharp_blur                 400                    200
DavidBowie train      400            flip_lr                 800                    400
DavidBowie train      400 flip_lr_sharp_blur                 800                    400
DavidBowie train      800            flip_lr                1600                    800
DavidBowie train      800 flip_lr_sharp_blur                1600                    800
    Endive train      200            flip_lr                 400                    200
    Endive train      200 flip_lr_sharp_blur                 400                    200
    Endive train      400            flip_lr                 800                    400
    Endive train      400 flip_lr_sharp_blur                 800                    400
    Endive train      800       

,bird,split,nframes,variant,unique_image_files,unique_logical_frames,labeled_data_dir
0,DavidBowie,train,200,flip_lr,400,200,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
1,DavidBowie,train,200,flip_lr_sharp_blur,400,200,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
2,DavidBowie,train,400,flip_lr,800,400,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
3,DavidBowie,train,400,flip_lr_sharp_blur,800,400,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
4,DavidBowie,train,800,flip_lr,1600,800,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
5,DavidBowie,train,800,flip_lr_sharp_blur,1600,800,..\DeepLabCut\DavidBowie\TestData\Canari_augex...
6,Endive,train,200,flip_lr,400,200,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...
7,Endive,train,200,flip_lr_sharp_blur,400,200,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...
8,Endive,train,400,flip_lr,800,400,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...
9,Endive,train,400,flip_lr_sharp_blur,800,400,..\DeepLabCut\Endive\TestData\Canari_augexp_fl...


## 8. Augmentation Experiment Matrix (200/400/800)

Train two augmentation variants per bird and frame count with explicit `modelprefix` labels:

- `flip_lr`
- `flip_lr_sharp_blur`

This cell runs one bodyparts confirmation per bird, then trains the augmentation matrix.

In [19]:
import shutil

In [26]:
AUGMENTATION_FRAME_COUNTS = [200, 400, 800]
RUN_AUGMENTATION_EXPERIMENT = True
USE_FULL_EPOCHS_AUG = False
AUG_EPOCHS = EPOCHS_FULL if USE_FULL_EPOCHS_AUG else EPOCHS_SMOKE

if "augmentation_variant_configs" not in globals():
    raise RuntimeError(
        "Run Cell 23 first so augmentation_variant_configs is available."
    )

# Keep these fixed for comparability.
AUG_SHUFFLE = 1
AUG_TRAININGSETINDEX = 0
AUG_GPUTOUSE = None

from deeplabcut.utils.auxiliaryfunctions import edit_config



# Use the same augmentation variants that were used to build project folders.
AUGMENTATION_VARIANTS = [
    {
        "name": "flip_lr",
        "modelprefix": "flip_lr",
        "pose_edits": {
            "fliplr": True,
            "symmetric_pairs": [(0, 14), (1, 12), (2, 13), (3, 11), (4, 9), (5, 10)],
            "sharpening": False,
            "blur": False,
        }
    },
    {
        "name": "flip_lr_sharp_blur",
        "modelprefix": "flip_lr_sharp_blur",
        "pose_edits": {
            "fliplr": True,
            "symmetric_pairs": [(0, 14), (1, 12), (2, 13), (3, 11), (4, 9), (5, 10)],
            "sharpening": True,
            "sharpenratio": 0.3,
            "blur": True,
            "blurratio": 0.3
        }
    },
]



def repair_video_set_labeled_data(config_path: Path) -> None:
    project_dir = Path(config_path).parent
    labeled_data_dir = project_dir / "labeled-data"

    # source folder that already has CollectedData
    source_h5 = sorted(labeled_data_dir.glob("*/CollectedData_*.h5"))
    if not source_h5:
        raise FileNotFoundError(f"No CollectedData_*.h5 found under {labeled_data_dir}")
    src_h5 = source_h5[0]
    src_csv = src_h5.with_suffix(".csv")

    with open(config_path, "r", encoding="utf-8") as fh:
        cfg = yaml.safe_load(fh)

    # For each video in video_sets, ensure matching labeled-data/<video_stem>/CollectedData exists
    for video_path in cfg.get("video_sets", {}).keys():
        stem = Path(str(video_path)).stem  # DB17_cam1
        dst_dir = labeled_data_dir / stem
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst_h5 = dst_dir / src_h5.name
        if not dst_h5.exists():
            shutil.copy2(src_h5, dst_h5)

        if src_csv.exists():
            dst_csv = dst_dir / src_csv.name
            if not dst_csv.exists():
                shutil.copy2(src_csv, dst_csv)


def apply_variant_pose_edits(config_path: Path, variant: dict) -> None:
    """Patch pose_cfg after DLC creates the training dataset for this variant."""
    pose_cfg_path, _, _ = deeplabcut.return_train_network_path(
        as_posix_str(config_path),
 #       shuffle=AUG_SHUFFLE,
   #     trainingsetindex=AUG_TRAININGSETINDEX,
        modelprefix=variant["modelprefix"],
    )
    print(f"pose config path: {pose_cfg_path}")
    pose_cfg_path = Path(pose_cfg_path)
    if not pose_cfg_path.exists():
        raise FileNotFoundError(f"pose_cfg.yaml not found at expected path: {pose_cfg_path}")

    edit_config(str(pose_cfg_path), variant["pose_edits"])
    

    with open(pose_cfg_path, "r", encoding="utf-8") as fh:
        pose_cfg = yaml.safe_load(fh)

    if variant["name"] == "flip_lr_sharp_blur":
        if not bool(pose_cfg.get("sharpening", False)) or not bool(pose_cfg.get("blur", False)):
            raise RuntimeError(
                f"Variant {variant['name']} requires both sharpening and blur in pose_cfg."
            )


augmentation_training_results = []

if not RUN_AUGMENTATION_EXPERIMENT:
    print("Dry run: set RUN_AUGMENTATION_EXPERIMENT = True to execute augmentation training.")
else:
    # Confirm bodyparts once per bird for all augmentation-specific configs.
    aug_review_configs_by_bird = {
        bird_name: [
            augmentation_variant_configs[(bird_name, n, variant["name"])]
            for n in AUGMENTATION_FRAME_COUNTS
            for variant in AUGMENTATION_VARIANTS
        ]
        for bird_name in BIRDS
    }
   # dlcs.require_bodyparts_review_before_training(
   #     run_training=True,
   #     configs_by_bird=aug_review_configs_by_bird,
   # )

    for bird_name in BIRDS:
        for n in AUGMENTATION_FRAME_COUNTS:
            # Keep architecture configurable through the existing notebook constant.
            for arch in ARCHITECTURES:
                for variant in AUGMENTATION_VARIANTS:
                    train_cfg = augmentation_variant_configs[(bird_name, n, variant["name"])]
                    run_label = f"{bird_name} | n={n} | {arch} | {variant['name']}"
                    print(f"\n{'─'*60}")
                    print(f"Augmentation training: {run_label}")

                    # Ensure architecture is set in the variant-specific config.
                    set_net_type(train_cfg, arch)

                    t0 = time.perf_counter()
                    try:
                        print('here')
                        repair_video_set_labeled_data(train_cfg)
                        print('here')
                        ensure_config_scorer_matches_data(train_cfg)
                        print('here')
                        deeplabcut.create_training_dataset(as_posix_str(train_cfg))  
                        
                        
                        print('here')
                        
                        apply_variant_pose_edits(train_cfg, variant)
                        # jump out of loop to avoid running multiple trainings while debugging pose edits
                        latest_snap = deeplabcut.train_network(as_posix_str(train_cfg),
                                                               epochs=AUG_EPOCHS,
                                                               snapshot_path=None,
                                                               modelprefix=variant["modelprefix"])
                        

                        
                        elapsed = time.perf_counter() - t0

                        augmentation_training_results.append({
                            "bird": bird_name,
                            "nframes": n,
                            "architecture": arch,
                            "augmentation": variant["name"],
                            "modelprefix": variant["modelprefix"],
                            "train_config": str(train_cfg),
                            "trained": True,
                            "train_time_s": round(elapsed, 1),
                            "latest_snapshot": latest_snap,
                            "notes": "",
                        })
                        print(f"  Done in {elapsed/60:.1f} min  ->  {latest_snap}")
                    except Exception as exc:
                        elapsed = time.perf_counter() - t0
                        augmentation_training_results.append({
                            "bird": bird_name,
                            "nframes": n,
                            "architecture": arch,
                            "augmentation": variant["name"],
                            "modelprefix": variant["modelprefix"],
                            "train_config": str(train_cfg),
                            "trained": False,
                            "train_time_s": round(elapsed, 1),
                            "latest_snapshot": None,
                            "notes": str(exc),
                        })
                        print(f"  ERROR: {exc}")

print(f"\n{'─'*60}")
print(f"Augmentation runs recorded: {len(augmentation_training_results)}")
if augmentation_training_results:
    pd.DataFrame(augmentation_training_results)[
        ["bird", "nframes", "architecture", "augmentation", "trained", "train_time_s", "train_config", "notes"]
    ]
else:
    print("No augmentation runs executed yet.")

INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead



────────────────────────────────────────────────────────────
Augmentation training: DavidBowie | n=200 | resnet_50 | flip_lr
here
here
here
  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\TestData\\Canari_augexp_flip_lr_n200\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation training: DavidBowie | n=200 | resnet_50 | flip_lr_sharp_blur
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


here
  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\DavidBowie\\TestData\\Canari_augexp_flip_lr_sharp_blur_n200\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation training: DavidBowie | n=400 | resnet_50 | flip_lr
here
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead
INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


...

here
  ERROR: Couldn't find any shuffles with trainingsetindex=0, shuffle=1 and modelprefix=flip_lr. Please check that such a shuffle is defined.

────────────────────────────────────────────────────────────
Augmentation training: Endive | n=400 | resnet_50 | flip_lr_sharp_blur
here
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\Endive\\TestData\\Canari_augexp_flip_lr_sharp_blur_n400\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation training: Endive | n=800 | resnet_50 | flip_lr
here
here
here


INFO | Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


here
  ERROR: Couldn't find any shuffles with trainingsetindex=0, shuffle=1 and modelprefix=flip_lr. Please check that such a shuffle is defined.

────────────────────────────────────────────────────────────
Augmentation training: Endive | n=800 | resnet_50 | flip_lr_sharp_blur
here
here
here
  ERROR: [Errno 2] No such file or directory: 'C:\\Users\\Salle-Cineradio\\Documents\\MachineLearning\\BirdSongs-MNHN\\Testing\\DeepLabCut\\Endive\\TestData\\Canari_augexp_flip_lr_sharp_blur_n800\\Canari-Tyler-2026-04-17\\training-datasets\\iteration-0\\UnaugmentedDataSet_CanariApr17\\Documentation_data-Canari_95shuffle1.pickle'

────────────────────────────────────────────────────────────
Augmentation runs recorded: 12


### Model Manipulation

How are we able to pull and train specific layers of the model to improve/transfer